In [1]:
import re
import math
from collections import Counter
import pandas as pd

In [3]:
# Data latih dan Data Uji
train_data = [
    ("Barang bagus dan cepat sampai", 5),
    ("Barang bagus sesuai deskripsi", 4),
    ("Barang cukup bagus sesuai harga", 3),
    ("Pesanan terlambat dan tidak sesuai", 2),
    ("Barang rusak dan tidak sesuai pesanan", 1)
]

test_data = "barang tidak sesuai"

print("\nData Latih")
for i, (text, rating) in enumerate(train_data):
    print(f"D{i+1} (Rating {rating}) : {text}")

print("\nData uji")
print(f"Dtest : {test_data}")


Data Latih
D1 (Rating 5) : Barang bagus dan cepat sampai
D2 (Rating 4) : Barang bagus sesuai deskripsi
D3 (Rating 3) : Barang cukup bagus sesuai harga
D4 (Rating 2) : Pesanan terlambat dan tidak sesuai
D5 (Rating 1) : Barang rusak dan tidak sesuai pesanan

Data uji
Dtest : barang tidak sesuai


In [5]:
# Preprocessing
slang_dict = {
    'ga':'tidak',
    'yg':'yang',
    'dgn':'dengan',
}

stopwords = {
    'dan',
    'yang',
    'dengan',
    'di',
    'ke',
    'dari',
    'untuk',
    'cukup'
}

def preprocess(text):
    # case folding
    text = text.lower()

    # cleansing
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # tokenisasi
    tokens = text.split()

    # slang normalization
    tokens = [slang_dict.get(token, token)
              for token in tokens]
    
    # stopword removal
    tokens = [token for token in tokens
              if token not in stopwords]
    
    return tokens

print("Hasil Preprocessing")
docs_tokens = []

for i, (text, rating) in enumerate(train_data):

    tokens = preprocess(text)
    docs_tokens.append(tokens)

    print(f"\nD{i+1}")
    print("Raw   :", text)
    print("Clean :", tokens)

test_tokens = preprocess(test_data)

print("\nDtest")
print("Raw   :", test_data)
print("Clean :", test_tokens)

Hasil Preprocessing

D1
Raw   : Barang bagus dan cepat sampai
Clean : ['barang', 'bagus', 'cepat', 'sampai']

D2
Raw   : Barang bagus sesuai deskripsi
Clean : ['barang', 'bagus', 'sesuai', 'deskripsi']

D3
Raw   : Barang cukup bagus sesuai harga
Clean : ['barang', 'bagus', 'sesuai', 'harga']

D4
Raw   : Pesanan terlambat dan tidak sesuai
Clean : ['pesanan', 'terlambat', 'tidak', 'sesuai']

D5
Raw   : Barang rusak dan tidak sesuai pesanan
Clean : ['barang', 'rusak', 'tidak', 'sesuai', 'pesanan']

Dtest
Raw   : barang tidak sesuai
Clean : ['barang', 'tidak', 'sesuai']


In [6]:
# Vocabulary
all_terms = sorted(
    list(
        set(
            term 
            for doc in docs_tokens
            for term in doc
        )
    )
)

print("Vocabulary")
print(all_terms)
V = len(all_terms)
print("\nJumlah vocabulary =", V)

Vocabulary
['bagus', 'barang', 'cepat', 'deskripsi', 'harga', 'pesanan', 'rusak', 'sampai', 'sesuai', 'terlambat', 'tidak']

Jumlah vocabulary = 11


In [7]:
# Term Frequency (TF)
print("Term Frequency (TF)")

tf_dict = {}
for i, doc in enumerate(docs_tokens):
    doc_name = f"D{i+1}"
    tf_dict[doc_name] = {}
    counter = Counter(doc)

    for term in all_terms:
        tf_dict[doc_name][term] = counter[term]

for doc_name in tf_dict:
    print(f"\n{doc_name}")

    for term in all_terms:
        if tf_dict[doc_name][term] > 0:
            print(
                f"{term:15} = {tf_dict[doc_name][term]}"
            )

Term Frequency (TF)

D1
bagus           = 1
barang          = 1
cepat           = 1
sampai          = 1

D2
bagus           = 1
barang          = 1
deskripsi       = 1
sesuai          = 1

D3
bagus           = 1
barang          = 1
harga           = 1
sesuai          = 1

D4
pesanan         = 1
sesuai          = 1
terlambat       = 1
tidak           = 1

D5
barang          = 1
pesanan         = 1
rusak           = 1
sesuai          = 1
tidak           = 1


In [8]:
# Document Frequency (DF)
print("Document Frequency (DF)")

df_dict = {}

for term in all_terms:
    df = 0

    for doc in docs_tokens:
        if term in doc:
            df +=1

    df_dict[term] = df

for term in all_terms:
    print(f"{term:15} -> df = {df_dict[term]}")

Document Frequency (DF)
bagus           -> df = 3
barang          -> df = 4
cepat           -> df = 1
deskripsi       -> df = 1
harga           -> df = 1
pesanan         -> df = 2
rusak           -> df = 1
sampai          -> df = 1
sesuai          -> df = 4
terlambat       -> df = 1
tidak           -> df = 2


In [9]:
# Inverse Document Frequency (IDF)
print("Inverse Document Frequency (IDF)")

N = len(docs_tokens)

idf_dict = {}

for term in all_terms:
    idf = math.log10(
        N / df_dict[term]
    )

    idf_dict[term] =  round(idf,4)

for term in all_terms:
    print(
        f"idf({term}) = "
        f"log10({N}/{df_dict[term]})"
        f"= {idf_dict[term]}"
    )

Inverse Document Frequency (IDF)
idf(bagus) = log10(5/3)= 0.2218
idf(barang) = log10(5/4)= 0.0969
idf(cepat) = log10(5/1)= 0.699
idf(deskripsi) = log10(5/1)= 0.699
idf(harga) = log10(5/1)= 0.699
idf(pesanan) = log10(5/2)= 0.3979
idf(rusak) = log10(5/1)= 0.699
idf(sampai) = log10(5/1)= 0.699
idf(sesuai) = log10(5/4)= 0.0969
idf(terlambat) = log10(5/1)= 0.699
idf(tidak) = log10(5/2)= 0.3979


In [26]:
# TF-IDF Data Latih
print("TF-IDF Data Latih")

tfidf_train = {}

for doc_name in tf_dict:
    tfidf_train[doc_name] = {}
    
    for term in all_terms:
        tfidf = (
            tf_dict[doc_name][term]
            *
            idf_dict[term]
        )

        tfidf_train[doc_name][term] = round(
            tfidf, 4
        )

for doc_name in tfidf_train:
    print(f"\n{doc_name}")
    
    for term in all_terms:
        if tfidf_train[doc_name][term] > 0:
            print(
                f"{term:15}"
                f"{tfidf_train[doc_name][term]}"
            )

TF-IDF Data Latih

D1
bagus          0.2218
barang         0.0969
cepat          0.699
sampai         0.699

D2
bagus          0.2218
barang         0.0969
deskripsi      0.699
sesuai         0.0969

D3
bagus          0.2218
barang         0.0969
harga          0.699
sesuai         0.0969

D4
pesanan        0.3979
sesuai         0.0969
terlambat      0.699
tidak          0.3979

D5
barang         0.0969
pesanan        0.3979
rusak          0.699
sesuai         0.0969
tidak          0.3979


In [27]:
# TF-IDF Data Uji
print("TF-IDF Data Uji")

counter_test = Counter(test_tokens)
tf_test = {}
tfidf_test =  {}

for term in all_terms:
    tf_test[term] = counter_test[term]
    tfidf_test[term] = round(
        tf_test[term]
        *
        idf_dict[term], 4
    )

for term in all_terms:
    if tf_test[term] > 0:
        print(
            f"{term:15}"
            f"TF={tf_test[term]}"
            f"IDF={idf_dict[term]}"
            f"TF-IDF={tfidf_test[term]}"
        )

TF-IDF Data Uji
barang         TF=1IDF=0.0969TF-IDF=0.0969
sesuai         TF=1IDF=0.0969TF-IDF=0.0969
tidak          TF=1IDF=0.3979TF-IDF=0.3979


In [28]:
# Prior Probability
print("Prior Probability")

ratings = [r for _, r in train_data]
class_counts = Counter(ratings)
prior = {}

for rating in sorted(class_counts):
    prior[rating] = (class_counts[rating]/len(train_data))

    print(
        f"P(Rating {rating})"
        f" = "
        f"{class_counts[rating]}/"
        f"{len(train_data)}"
        f" = "
        f"{prior[rating]:.3f}"
    )

Prior Probability
P(Rating 1) = 1/5 = 0.200
P(Rating 2) = 1/5 = 0.200
P(Rating 3) = 1/5 = 0.200
P(Rating 4) = 1/5 = 0.200
P(Rating 5) = 1/5 = 0.200


In [29]:
# Likelihood
print("Likelihood + Laplace Smoothing")

alpha = 1
likelihood = {}

for rating in sorted(class_counts):
    likelihood[rating] = {}
    class_docs = []

    for i, (_, r) in enumerate(train_data):
        if r == rating:
            class_docs.extend(docs_tokens[i])
    
    total_tokens = len(class_docs)
    counter = Counter(class_docs)
    print(f"\nRating {rating}")

    for term in all_terms:
        prob = (counter[term] + alpha)/(total_tokens + alpha * V)

        likelihood[rating][term] = prob
        print(
            f"P({term}|{rating})"
            f" = "
            f"({counter[rating]} + 1)"
            f"/"
            f"({total_tokens} + {V})"
            f" = "
            f"{prob:.4f}"
        )

Likelihood + Laplace Smoothing

Rating 1
P(bagus|1) = (0 + 1)/(5 + 11) = 0.0625
P(barang|1) = (0 + 1)/(5 + 11) = 0.1250
P(cepat|1) = (0 + 1)/(5 + 11) = 0.0625
P(deskripsi|1) = (0 + 1)/(5 + 11) = 0.0625
P(harga|1) = (0 + 1)/(5 + 11) = 0.0625
P(pesanan|1) = (0 + 1)/(5 + 11) = 0.1250
P(rusak|1) = (0 + 1)/(5 + 11) = 0.1250
P(sampai|1) = (0 + 1)/(5 + 11) = 0.0625
P(sesuai|1) = (0 + 1)/(5 + 11) = 0.1250
P(terlambat|1) = (0 + 1)/(5 + 11) = 0.0625
P(tidak|1) = (0 + 1)/(5 + 11) = 0.1250

Rating 2
P(bagus|2) = (0 + 1)/(4 + 11) = 0.0667
P(barang|2) = (0 + 1)/(4 + 11) = 0.0667
P(cepat|2) = (0 + 1)/(4 + 11) = 0.0667
P(deskripsi|2) = (0 + 1)/(4 + 11) = 0.0667
P(harga|2) = (0 + 1)/(4 + 11) = 0.0667
P(pesanan|2) = (0 + 1)/(4 + 11) = 0.1333
P(rusak|2) = (0 + 1)/(4 + 11) = 0.0667
P(sampai|2) = (0 + 1)/(4 + 11) = 0.0667
P(sesuai|2) = (0 + 1)/(4 + 11) = 0.1333
P(terlambat|2) = (0 + 1)/(4 + 11) = 0.1333
P(tidak|2) = (0 + 1)/(4 + 11) = 0.1333

Rating 3
P(bagus|3) = (0 + 1)/(4 + 11) = 0.1333
P(barang|3) = (0

In [30]:
# Posterior
print("Posterior Probability")

posterior = {}
for rating in sorted(class_counts):
    prob = prior[rating]
    print("\nRating {rating}")
    print(
        f"P({rating}|D)"
        f"∝"
        f"{prior[rating]:.4f}"
    )

    for term in test_tokens:
        if term in all_terms:
            prob *= likelihood[rating][term]
            print(
                f"x P({term}|{rating})"
                f" = "
                f"{likelihood[rating][term]:.4f}"
            )

    posterior[rating] = prob
    print(f"Posterior =  {prob:.8f}")

Posterior Probability

Rating {rating}
P(1|D)∝0.2000
x P(barang|1) = 0.1250
x P(tidak|1) = 0.1250
x P(sesuai|1) = 0.1250
Posterior =  0.00039063

Rating {rating}
P(2|D)∝0.2000
x P(barang|2) = 0.0667
x P(tidak|2) = 0.1333
x P(sesuai|2) = 0.1333
Posterior =  0.00023704

Rating {rating}
P(3|D)∝0.2000
x P(barang|3) = 0.1333
x P(tidak|3) = 0.0667
x P(sesuai|3) = 0.1333
Posterior =  0.00023704

Rating {rating}
P(4|D)∝0.2000
x P(barang|4) = 0.1333
x P(tidak|4) = 0.0667
x P(sesuai|4) = 0.1333
Posterior =  0.00023704

Rating {rating}
P(5|D)∝0.2000
x P(barang|5) = 0.1333
x P(tidak|5) = 0.0667
x P(sesuai|5) = 0.0667
Posterior =  0.00011852


In [34]:
# Hasil Klasifikasi
print("Hasil Klasifikasi")
print(f"Data Test: {test_data}\n")

for rating in sorted(posterior):
    print(
        f"Rating {rating}"
        f" : "
        f"{posterior[rating]:.8f}"  
    )

prediksi = max(posterior, key=posterior.get)
print("\nPrediksi Akhir :", prediksi)

Hasil Klasifikasi
Data Test: barang tidak sesuai

Rating 1 : 0.00039063
Rating 2 : 0.00023704
Rating 3 : 0.00023704
Rating 4 : 0.00023704
Rating 5 : 0.00011852

Prediksi Akhir : 1


In [36]:
# Table Akhir
rows = []

for term in sorted(all_terms):

    row = {
        'Term': term,
        'df': df_dict[term],
        'idf': round(idf_dict[term],3),

        'TF(D1)': tf_dict['D1'][term],
        'TF(D2)': tf_dict['D2'][term],
        'TF(D3)': tf_dict['D3'][term],
        'TF(D4)': tf_dict['D4'][term],
        'TF(D5)': tf_dict['D5'][term],

        'TF-IDF(D1)': round(tfidf_train['D1'][term],3),
        'TF-IDF(D2)': round(tfidf_train['D2'][term],3),
        'TF-IDF(D3)': round(tfidf_train['D3'][term],3),
        'TF-IDF(D4)': round(tfidf_train['D4'][term],3),
        'TF-IDF(D5)': round(tfidf_train['D5'][term],3),
    }

    rows.append(row)

df_table = pd.DataFrame(rows)

print(df_table.to_string(index=False))

     Term  df   idf  TF(D1)  TF(D2)  TF(D3)  TF(D4)  TF(D5)  TF-IDF(D1)  TF-IDF(D2)  TF-IDF(D3)  TF-IDF(D4)  TF-IDF(D5)
    bagus   3 0.222       1       1       1       0       0       0.222       0.222       0.222       0.000       0.000
   barang   4 0.097       1       1       1       0       1       0.097       0.097       0.097       0.000       0.097
    cepat   1 0.699       1       0       0       0       0       0.699       0.000       0.000       0.000       0.000
deskripsi   1 0.699       0       1       0       0       0       0.000       0.699       0.000       0.000       0.000
    harga   1 0.699       0       0       1       0       0       0.000       0.000       0.699       0.000       0.000
  pesanan   2 0.398       0       0       0       1       1       0.000       0.000       0.000       0.398       0.398
    rusak   1 0.699       0       0       0       0       1       0.000       0.000       0.000       0.000       0.699
   sampai   1 0.699       1       0     